In [1]:
!python -m spacy download fr_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 14.1 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

nlp = spacy.load("fr_core_news_sm")

train = pd.read_csv("/train.csv")
dev = pd.read_csv("/dev.csv")
test = pd.read_csv("/test.csv")

print("Train shape:", train.shape)
print("Columns:", train.columns.tolist())
print(f"First entry - polarity: {train['polarity'][0]}\nReview: {train['review'][0]}")
print("Dev shape:", dev.shape)
print("Test shape:", test.shape)

# 2. Generating tokens
def review_to_tokens(raw_review, all_tokens=True):
    doc = nlp(str(raw_review))
    if all_tokens:
        return " ".join([token.text for token in doc])
    else:
        return " ".join([token.text for token in doc if token.pos_ in ['VERB', 'NOUN']])

def preprocess_data(dataset, all_tokens=True):
    num_reviews = dataset['review'].size
    print("#Reviews =", num_reviews)
    dataset_tok = []

    # Convert reviews to a standard list of strings
    texts = dataset['review'].astype(str).tolist()

    # nlp.pipe processes data in optimized batches
    for doc in nlp.pipe(texts, disable=["parser", "ner"], batch_size=256):
        if all_tokens:
            clean_review = " ".join([token.text for token in doc])
        else:
            clean_review = " ".join([token.text for token in doc if token.pos_ in ['VERB', 'NOUN']])

        dataset_tok.append(clean_review)

    for i, r in enumerate(dataset_tok[:2]):
        print('\n', i, dataset['review'].iloc[i])
        print(r)

    return dataset_tok

# 3. Displaying preprocessed data
print("\n<<< Preprocessing with ALL tokens >>>")
train_tok_all = preprocess_data(train, all_tokens=True)
dev_tok_all = preprocess_data(dev, all_tokens=True)
test_tok_all = preprocess_data(test, all_tokens=True)

print("\n<<< Preprocessing with VERBS & NOUNS only >>>")
train_tok_vn = preprocess_data(train, all_tokens=False)
dev_tok_vn = preprocess_data(dev, all_tokens=False)
test_tok_vn = preprocess_data(test, all_tokens=False)

# 4 & 5. Vectorizing and defining the vocabulary
vectorizer = CountVectorizer()
X_train_all = vectorizer.fit_transform(train_tok_all)
X_dev_all = vectorizer.transform(dev_tok_all)

print("\nFirst 30 vocabulary words:")
print(list(vectorizer.vocabulary_.keys())[:30])

# 6. Restricting vocabulary to 5000
vectorizer_5000 = CountVectorizer(max_features=5000)
X_train_5000 = vectorizer_5000.fit_transform(train_tok_all)
X_dev_5000 = vectorizer_5000.transform(dev_tok_all)

# 7 & 8. Training Naive Bayes and Evaluation
nb_clf = MultinomialNB()
nb_clf.fit(X_train_5000, train['polarity'])
print("\nNB Dev Score (max_features=5000):", nb_clf.score(X_dev_5000, dev['polarity']))

pred = nb_clf.predict(X_dev_5000)
print("\nClassification Report (NB):")
print(classification_report(dev['polarity'], pred))

#9. Displaying misclassified instances
print('Misclassified examples (showing first 5): ')
count_err = 0
for i in range(len(pred)):
    if pred[i] != dev['polarity'][i]:
        if count_err < 5:
            print(f"\nGOLD = {dev['polarity'][i]} PRED= {pred[i]} {i} {dev['review'][i]}")
        count_err += 1
print(f"Total errors: {count_err}")

# 10. Bigrams (max_features=500)
vec_bigram = CountVectorizer(ngram_range=(2,2), max_features=500)
X_train_bi = vec_bigram.fit_transform(train_tok_all)
X_dev_bi = vec_bigram.transform(dev_tok_all)
nb_bi = MultinomialNB().fit(X_train_bi, train['polarity'])
print("\nNB Dev Score (Bigrams only, max_features=500):", nb_bi.score(X_dev_bi, dev['polarity']))

# 11. Verbs and Nouns only
vec_vn = CountVectorizer(max_features=5000)
X_train_vn = vec_vn.fit_transform(train_tok_vn)
X_dev_vn = vec_vn.transform(dev_tok_vn)
nb_vn = MultinomialNB().fit(X_train_vn, train['polarity'])
print("NB Dev Score (Verbs/Nouns only):", nb_vn.score(X_dev_vn, dev['polarity']))

# 12. Unigrams and Bigrams
vec_uni_bi = CountVectorizer(ngram_range=(1,2), max_features=5000)
X_train_uni_bi = vec_uni_bi.fit_transform(train_tok_all)
X_dev_uni_bi = vec_uni_bi.transform(dev_tok_all)
nb_uni_bi = MultinomialNB().fit(X_train_uni_bi, train['polarity'])
print("NB Dev Score (Unigrams & Bigrams, max=5000):", nb_uni_bi.score(X_dev_uni_bi, dev['polarity']))

# 13. Comparing with MaxEnt (Logistic Regression)
lr_clf = LogisticRegression(max_iter=1000)
lr_clf.fit(X_train_uni_bi, train['polarity'])
print("Logistic Regression Dev Score (Unigrams & Bigrams):", lr_clf.score(X_dev_uni_bi, dev['polarity']))

# 14. Performance on test set
X_test_uni_bi = vec_uni_bi.transform(test_tok_all)
print("\nLogistic Regression TEST Score:", lr_clf.score(X_test_uni_bi, test['polarity']))

# 15. Conclusion
print("\nConclusion:")
print("Combining Unigrams and Bigrams yields better contextual representation than restricting to just Verbs/Nouns or just Bigrams. Furthermore, Logistic Regression (MaxEnt) typically handles correlated features (like unigrams and bigrams combined) better than Naive Bayes, leading to stronger overall performance on both Dev and Test sets.")

Train shape: (160000, 4)
Columns: ['Unnamed: 0', 'film-url', 'review', 'polarity']
First entry - polarity: 0
Review: Si vous cherchez du cinéma abrutissant à tous les étages,n'ayant aucune peur du cliché en castagnettes et moralement douteux,"From Paris with love" est fait pour vous.Toutes les productions Besson,via sa filière EuropaCorp ont de quoi faire naître la moquerie.Paris y est encore une fois montrée comme une capitale exotique,mais attention si l'on se dirige vers la banlieue,on y trouve tout plein d'intégristes musulmans prêts à faire sauter le caisson d'une ambassadrice américaine.Nauséeux.Alors on se dit qu'on va au moins pouvoir apprécier la déconnade d'un classique buddy-movie avec le jeune agent aux dents longues obligé de faire équipe avec un vieux lou complètement timbré.Mais d'un côté,on a un Jonathan Rhys-meyers fayot au possible,et de l'autre un John Travolta en total délire narcissico-badass,crâne rasé et bouc proéminent à l'appui.Sinon,il n'y a aucun scénario.Seu